In [15]:
import pandas as pd
import numpy as np
import os


In [16]:
def is_date_column(series):
    return pd.api.types.is_datetime64_any_dtype(series)

def is_coref_column(series, threshold=50):
    num_unique = series.nunique(dropna=True)
    return series.dtype == 'object' and num_unique > threshold

def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if series.dtype == 'object':
        try:
            parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
            if all(parsed.dt.time == pd.Timestamp("00:00:00").time()):
                return "date"
            else:
                return "datetime"
        except:
            pass

        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return "time"
        except:
            pass

        return "categorical"

    return "categorical"


In [17]:
def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if is_date_column(series) or is_coref_column(series):
        return []

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f"{series.min()} - {series.max()}"

    return unique_vals.tolist()


In [18]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "null_count": series.isna().sum(),
            "null_percentage": round((series.isna().sum() / len(df)) * 100, 2),
            "num_unique": series.nunique(dropna=True),
            "unique_values": get_unique_representation(series),
            "top_values": series.value_counts(dropna=True).head(5).to_dict(),
            "skewness": series.skew() if pd.api.types.is_numeric_dtype(series) else None,
            "constant_column": series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)


In [ ]:
def analyze_dataset(file_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(file_path)

    print(f"\nProcessing: {file_path}")
    print(f"Columns: {len(df.columns)}")

    df_cols = pd.DataFrame(df.columns, columns=['Columns'])
    df_cols.to_csv(os.path.join(output_dir, 'columns.csv'), index=False)

    null_columns = df.columns[df.isnull().any()]
    null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns'])
    null_cols_df.to_csv(os.path.join(output_dir, 'null_columns.csv'), index=False)

    df_validation = create_data_validation_df(df)
    df_validation.to_csv(os.path.join(output_dir, 'data_validation_summary.csv'), index=False)

    print("Analysis complete\n")


In [ ]:
analyze_dataset(
    file_path='./dataset/01_raw/billings.csv',
    output_dir='./dataset/01_raw/ad_hoc/'
)


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\263505632.py:4: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)



Processing: ./dataset/01_raw/billings.csv
Columns: 59


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_462

Analysis complete



In [21]:
files = [
    './dataset/01_raw/billings.csv',
    './dataset/01_raw/cc_calls.csv',
    './dataset/01_raw/renewal_calls.csv',
    './dataset/01_raw/emails.csv'
    
]

for file in files:
    name = file.split('/')[-1].replace('.csv', '')
    output_dir = f'./dataset/01_raw/ad_hoc/{name}/'
    analyze_dataset(file, output_dir)


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\263505632.py:4: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')



Processing: ./dataset/01_raw/billings.csv
Columns: 59


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_462

Analysis complete


Processing: ./dataset/01_raw/cc_calls.csv
Columns: 33


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_462

...

Analysis complete



C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\263505632.py:4: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')



Processing: ./dataset/01_raw/emails.csv
Columns: 27


C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_4620\2152732644.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_462

Analysis complete

